In [ ]:
import torch
import torch.nn as nn 
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
import pandas as pd
import os
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from PIL import Image
from torch.utils.data import random_split
from tqdm.auto import tqdm
import optuna

In [43]:
label_names = [
    "액션", "코믹", "드라마", "호러", "SF", "로맨스",      # 모델 1 (레이블 1~6)
    "낮", "밤", "실내", "실외", "도시", "자연"            # 모델 2 (레이블 7~12)
]

In [54]:
# 1. 하이퍼파라미터 및 경로 설정
CSV_PATH = "data/datasets/씬스틸러(SceneStealer)_DATA_LABELS_TOTAL.csv"
IMG_DIR = "data/datasets/images"
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-4  # EfficientNet은 작은 학습률에서 안정적으로 학습됩니다.

In [86]:
df = pd.read_csv(CSV_PATH)
print(df.columns)

Index(['파일명', '문화유산', '도심', '상점가', '공원', '산', '물가', '데이트/로맨틱', '힐링/여유',
       '럭셔리/고급', '액티브/아웃도어', '가족/키즈', '야경/밤감성'],
      dtype='object')


In [92]:
cols_part1 = df.columns[1:7]
cols_part2 = df.columns[7:]

In [55]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [96]:
class MultiLabelDataset(Dataset):
 
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        #print(self.data.head(1))
        self.img_dir = img_dir
        self.transform = transform
        #self.label_range = label_range

        # CSV 컬럼명에 맞춰 조정 필요 (예: ['label1', ..., 'label6'])
        self.cols_part1 = cols_part1
        self.cols_part2 = cols_part2

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['파일명']
        img_path = f"{self.img_dir}/{img_name}"
        image = Image.open(img_path).convert('RGB')
        
        labels1 = torch.tensor(self.df.iloc[idx][self.cols_part1].values.astype(float), dtype=torch.float32)
        labels2 = torch.tensor(self.df.iloc[idx][self.cols_part2].values.astype(float), dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
            
        return image, labels1, labels2

In [97]:
# 2. 데이터 증강 (Best 모델을 위한 필수 단계)
# 학습 데이터에는 변형을 주어 일반화 성능을 높이고, 검증 데이터는 원본 유지
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), # 좌우 반전
    transforms.RandomRotation(10),      # 살짝 회전
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # 밝기/대비 조절
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3. 데이터셋 분할
# 이미 정의하신 MultiLabelDataset을 사용한다고 가정합니다.
dataset = MultiLabelDataset(csv_file=CSV_PATH, img_dir=IMG_DIR, transform=train_transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_data, val_data = random_split(dataset, [train_size, val_size])

# 검증 셋에는 변형이 없는 val_transform 적용 (중요)
val_data.dataset.transform = val_transform

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)

# 4. 모델 생성 함수 (EfficientNet-B0)
def create_model(num_classes):
    # ImageNet으로 사전 학습된 가중치를 베이스로 사용하여 성능을 극대화합니다.
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    # 출력 레이어를 사용자의 레이블 개수(6개)에 맞게 변경
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model.to(DEVICE)

# 5. 학습 핵심 루틴 (Train & Validate)
def run_training(target_part, model_save_path, train_loader, val_loader):
    print(f"\n🚀 모델 {target_part} 학습 시작")
    
    model = create_model(num_classes=6)
    criterion = nn.BCEWithLogitsLoss() # 멀티레이블 분류의 핵심
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)
    
    best_loss = float('inf')
    early_stop_patience = 7
    early_stop_counter = 0

    for epoch in range(EPOCHS):
        # --- Training ---
        model.train()
        train_loss = 0
        for imgs, labels1, labels2 in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            imgs = imgs.to(DEVICE)
            # 타겟 파트에 따라 레이블 선택
            targets = labels1.to(DEVICE) if target_part == 1 else labels2.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # --- Validation ---model_save_patht
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for imgs, labels1, labels2 in val_loader:
                imgs = imgs.to(DEVICE)
                targets = labels1.to(DEVICE) if target_part == 1 else labels2.to(DEVICE)
                outputs = model(imgs)
                val_loss += criterion(outputs, targets).item()

        avg_val_loss = val_loss / len(val_loader)
        
        # 학습률 조절
        scheduler.step(avg_val_loss)
        
        # Best 모델 저장 (최저 Loss 기준)
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            torch.save(model.state_dict(), model_save_path)
            print(f"✅ Best Model Saved: {model_save_path}")
            early_stop_counter = 0
        else:
            early_stop_counter += 1

        if early_stop_counter >= early_stop_patience:
            break

        


In [99]:

# 6. 실제 학습 실행
#if __name__ == "__main__":
    # 데이터 로더가 위에서 정의되어 있어야 합니다.
    # train_loader = DataLoader(...)
    # val_loader = DataLoader(...)

    # 함수 호출 시 정의한 loader를 넘겨줍니다.
    
run_training(1, "best_effic_model_part1.pth", train_loader, val_loader)
run_training(2, "best_effic_model_part2.pth", train_loader, val_loader)


🚀 모델 1 학습 시작


Epoch 1/10:   0%|          | 0/24 [00:00<?, ?it/s]

c:\potenup3\prj_deep\.venv\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (103038840 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Epoch 2/10:   0%|          | 0/24 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# 2. 전처리 설정 (학습 시 사용한 Resolution과 동일해야 함)
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# ])

# dataset = MultiLabelDataset(
#     csv_file="data/datasets/씬스틸러(SceneStealer)_DATA_LABELS_TOTAL.csv",
#     img_dir="data/datasets/images",
#     transform=transform
# )

In [ ]:
# # 1. 데이터 로더 설정 (80% 학습, 20% 검증 분할 추천)
# train_size = int(0.8 * len(dataset))
# val_size = len(dataset) - train_size
# train_db, val_db = torch.utils.data.random_split(dataset, [train_size, val_size])

In [ ]:
# train_loader = DataLoader(train_db, batch_size=32, shuffle=True)
# val_loader = DataLoader(val_db, batch_size=32, shuffle=False)

In [ ]:
# num_classes = 6

# EfficientNet 모델 

In [ ]:
# 2. 모델 1 학습 (레이블 1~6용)
# model1 = get_efficientnet_model(num_classes=6)
# train_model(model1, train_loader, val_loader, model_name="scene_stealer_part1.pth")

# 3. 모델 2 학습 (레이블 7~12용) - 위 train_model 함수 내부의 targets만 수정하여 실행
# model2 = get_efficientnet_model(num_classes=6)
# train_model(model2, train_loader, val_loader, model_name="scene_stealer_part2.pth")

In [ ]:
# def train_model(model, train_loader, val_loader, model_name="best_model.pth", epochs=30, lr=0.001):
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     model.to(device)
    
#     # 멀티레이블 분류를 위한 손실 함수와 옵티마이저
#     criterion = nn.BCEWithLogitsLoss()
#     optimizer = optim.Adam(model.parameters(), lr=lr)
    
#     # 학습률 스케줄러: 5번의 에폭 동안 성능 향상이 없으면 학습률을 0.1배로 감소
#     scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.1)
    
#     best_val_loss = float('inf')
#     early_stop_count = 0
#     patience_limit = 10  # 10번 연속 개선 없으면 조기 종료
    
#     for epoch in range(epochs):
#         # --- Training Phase ---
#         model.train()
#         train_loss = 0
#         for images, labels1, labels2 in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
#             # 현재 학습하려는 모델의 타겟 레이블 선택 (예: labels1 또는 labels2)
#             # 여기서는 편의상 labels1을 예시로 함 (외부에서 타겟 결정 필요)
#             targets = labels1.to(device) 
#             images = images.to(device)
            
#             optimizer.zero_grad()
#             outputs = model(images)
#             loss = criterion(outputs, targets)
#             loss.backward()
#             optimizer.step()
#             train_loss += loss.item()
            
#         # --- Validation Phase ---
#         model.eval()
#         val_loss = 0
#         with torch.no_grad():
#             for images, labels1, labels2 in val_loader:
#                 targets = labels1.to(device) 
#                 images = images.to(device)
#                 outputs = model(images)
#                 loss = criterion(outputs, targets)
#                 val_loss += loss.item()
        
#         avg_train_loss = train_loss / len(train_loader)
#         avg_val_loss = val_loss / len(val_loader)
        
#         print(f"Epoch [{epoch+1}] Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
        
#         # 스케줄러 업데이트
#         scheduler.step(avg_val_loss)
        
#         # 베스트 모델 저장 (Checkpoint)
#         if avg_val_loss < best_val_loss:
#             best_val_loss = avg_val_loss
#             torch.save(model.state_dict(), model_name)
#             print(f"--> Best Model Saved: {model_name}")
#             early_stop_count = 0
#         else:
#             early_stop_count += 1
            
#         # Early Stopping 조기 종료
#         if early_stop_count >= patience_limit:
#             print("Early stopping triggered. Training stopped.")
#             break
            
#     print("Training Complete.")

In [ ]:
# def get_efficientnet_model(num_classes):
#     # EfficientNet B0 사용 (필요시 Bx로 변경)
#     model = models.efficientnet_b0(pretrained=True)
#     model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
#     return model

# # 모델 2개 생성
# model1 = get_efficientnet_model(num_classes=6) # 1~6
# model2 = get_efficientnet_model(num_classes=6) # 7~12

c:\potenup3\prj_deep\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\potenup3\prj_deep\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
# import matplotlib.pyplot as plt
# import torch.nn.functional as F

# def predict_and_visualize(image_path, model1, model2, label_names):
#     # 1. 전처리 (학습 시와 동일한 transform 적용)
#     image = Image.open(image_path).convert('RGB')
#     input_tensor = transform(image).unsqueeze(0) 
    
#     # 2. 모델 추론
#     model1.eval()
#     model2.eval()
#     with torch.no_grad():
#         out1 = torch.sigmoid(model1(input_tensor)) # Multi-label은 Sigmoid
#         out2 = torch.sigmoid(model2(input_tensor))
        
#     # 3. 결과 결합
#     all_probs = torch.cat([out1, out2], dim=1).squeeze(0)
    
#     # 4. Top 5 추출
#     top5_prob, top5_idx = torch.topk(all_probs, 5)
    
#     # 5. 시각화
#     plt.imshow(image)
#     plt.axis('off')
#     plt.title("Top 5 Predictions")
#     plt.show()
    
#     for i in range(5):
#         print(f"{i+1}위: {label_names[top5_idx[i]]} (확률: {top5_prob[i]:.2%})")

# label_names는 [label1_name, ..., label12_name] 리스트

# VGG16 모델

In [ ]:
# VGG16 모델
#model = models.vgg16(pretrained=True)
model = models.vgg16(weights='DEFAULT')

model.classifier[6] = nn.Linear(4096, num_classes)

model = model.cuda()

In [14]:
criterion=nn.BCEWithLogitsLoss()

## 학습
## VGG16 멀티레이블 모델
#### VGG16 마지막 layer만 수정합니다.

In [15]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

dataset = MultiLabelDataset(
    csv_file="data/datasets/씬스틸러(SceneStealer)_DATA_LABELS_TOTAL.csv",
    img_dir="data/datasets/images",
    transform=transform
)


In [16]:

train_loader = DataLoader(dataset,
                          batch_size=16,
                          shuffle=True)

In [30]:

# Loss
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 10
for epoch in range(EPOCHS):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        label = lablabels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, label)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} Loss {running_loss:.4f}")

torch.save(model.state_dict(),"multilabel_vgg16.pth")


NameError: name 'label' is not defined

## 추론
##### 멀티레이블은 sigmoid 후 threshold 사용합니다.

In [ ]:
model.eval()

with torch.no_grad():

    output = model(image.unsqueeze(0).cuda())

    probs = torch.sigmoid(output)

    preds = (probs > 0.5).int()

print(preds)

## 이름 매핑

In [ ]:
classes= ['A','B','C','D','E','F','G']

pred_labels= [classes[i] for i,vinenumerate(preds[0]) if v==1]

print(pred_labels)